In [ ]:

!pip install -q scikit-surprise nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.8 MB/s eta 0:00:00


In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import ast
import re
import pickle
import warnings

warnings.filterwarnings("ignore")

from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import normalize

from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import csr_matrix
from scipy.sparse import hstack

from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

movies = pd.read_csv(
    "/content/drive/MyDrive/new_movie_dataset/movies.csv"
)

ratings = pd.read_csv(
    "/content/drive/MyDrive/new_movie_dataset/ratings.csv"
)

tmdb_movies = pd.read_csv(
    "/content/drive/MyDrive/new_movie_dataset/tmdb_5000_movies.csv"
)

credits = pd.read_csv(
    "/content/drive/MyDrive/new_movie_dataset/tmdb_5000_credits.csv"
)

In [ ]:
print("Movies :", movies.shape)
print("Ratings :", ratings.shape)
print("TMDB :", tmdb_movies.shape)
print("Credits :", credits.shape)

Movies : (10329, 3)
Ratings : (105339, 4)
TMDB : (4803, 20)
Credits : (4803, 4)


In [ ]:
print(movies.isnull().sum())

print(ratings.isnull().sum())

print(tmdb_movies.isnull().sum())

print(credits.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64
movie_id    0
title       0
cast        0
crew        0
dtype: int64


In [ ]:
movies.drop_duplicates(inplace=True)

ratings.drop_duplicates(inplace=True)

tmdb_movies.drop_duplicates(inplace=True)

credits.drop_duplicates(inplace=True)

In [ ]:
movies["clean_title"] = movies["title"].str.replace(
    r"\(\d{4}\)",
    "",
    regex=True
)

movies["clean_title"] = movies["clean_title"].str.strip()

In [ ]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")
movies["year"] = movies["year"].astype(float)

In [ ]:
tmdb_movies["year"] = pd.to_datetime(
    tmdb_movies["release_date"],
    errors="coerce"
).dt.year

In [ ]:
tmdb = tmdb_movies.merge(
    credits,
    left_on="id",
    right_on="movie_id"
)

In [ ]:
print(tmdb.shape)

(4803, 25)


In [ ]:
print(tmdb.columns.tolist())

['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average', 'vote_count', 'year', 'movie_id', 'title_y', 'cast', 'crew']


In [ ]:
tmdb = tmdb_movies.merge(
    credits,
    left_on="id",
    right_on="movie_id",
    how="inner"
)

# Rename columns for clarity
tmdb.rename(columns={
    "title_x": "title_tmdb",
    "title_y": "title_credit"
}, inplace=True)

print(tmdb.columns)

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_tmdb', 'vote_average',
       'vote_count', 'year', 'movie_id', 'title_credit', 'cast', 'crew'],
      dtype='object')


In [ ]:
merged = movies.merge(
    tmdb,
    left_on=["clean_title", "year"],
    right_on=["title_tmdb", "year"],
    how="inner"
)

In [ ]:
print(merged.columns.tolist())

['movieId', 'title', 'genres_x', 'clean_title', 'year', 'budget', 'genres_y', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title_tmdb', 'vote_average', 'vote_count', 'movie_id', 'title_credit', 'cast', 'crew']


In [ ]:


merged.rename(columns={
    "title": "title_ml",
    "genres_x": "genres_ml",
    "genres_y": "genres_tmdb",
    "id": "tmdb_id"
}, inplace=True)

In [ ]:
print(merged.columns.tolist())

['movieId', 'title_ml', 'genres_ml', 'clean_title', 'year', 'budget', 'genres_tmdb', 'homepage', 'tmdb_id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title_tmdb', 'vote_average', 'vote_count', 'movie_id', 'title_credit', 'cast', 'crew']


In [ ]:
merged = merged[
    [
        "movieId",
        "tmdb_id",
        "title_ml",
        "title_tmdb",
        "clean_title",
        "year",
        "genres_ml",
        "genres_tmdb",
        "keywords",
        "overview",
        "cast",
        "crew",
        "popularity",
        "vote_average",
        "vote_count"
    ]
]

In [ ]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2524 entries, 0 to 2523
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movieId       2524 non-null   int64  
 1   tmdb_id       2524 non-null   int64  
 2   title_ml      2524 non-null   object 
 3   title_tmdb    2524 non-null   object 
 4   clean_title   2524 non-null   object 
 5   year          2524 non-null   float64
 6   genres_ml     2524 non-null   object 
 7   genres_tmdb   2524 non-null   object 
 8   keywords      2524 non-null   object 
 9   overview      2524 non-null   object 
 10  cast          2524 non-null   object 
 11  crew          2524 non-null   object 
 12  popularity    2524 non-null   float64
 13  vote_average  2524 non-null   float64
 14  vote_count    2524 non-null   int64  
dtypes: float64(3), int64(3), object(9)
memory usage: 295.9+ KB


In [ ]:

import ast

In [ ]:

def extract_names(text):

    try:
        items = ast.literal_eval(text)

        return [item["name"] for item in items]

    except:

        return []

In [ ]:
merged["genres_tmdb"] = merged["genres_tmdb"].apply(extract_names)

In [ ]:
merged["genres_tmdb"].head()

,genres_tmdb
0,"[Animation, Comedy, Family]"
1,"[Adventure, Action, Thriller]"
2,"[History, Drama]"
3,"[Action, Adventure]"
4,"[Drama, Crime]"


In [ ]:
merged["keywords"] = merged["keywords"].apply(extract_names)

In [ ]:
def extract_cast(text):

    try:

        cast = ast.literal_eval(text)

        actors = []

        for person in cast[:3]:

            actors.append(
                person["name"].replace(" ", "")
            )

        return actors

    except:

        return []

In [ ]:
merged["cast"] = merged["cast"].apply(extract_cast)

In [ ]:
merged["cast"].head()

,cast
0,"[TomHanks, TimAllen, DonRickles]"
1,"[PierceBrosnan, SeanBean, IzabellaScorupco]"
2,"[AnthonyHopkins, JoanAllen, PowersBoothe]"
3,"[GeenaDavis, MatthewModine, FrankLangella]"
4,"[RobertDeNiro, SharonStone, JoePesci]"


In [ ]:
def extract_director(text):

    try:

        crew = ast.literal_eval(text)

        for member in crew:

            if member["job"] == "Director":

                return member["name"].replace(" ", "")

    except:

        pass

    return ""

In [ ]:
merged["director"] = merged["crew"].apply(extract_director)

In [ ]:
merged["director"].head()

,director
0,JohnLasseter
1,MartinCampbell
2,OliverStone
3,RennyHarlin
4,MartinScorsese


In [ ]:
merged["overview"] = merged["overview"].fillna("")

In [ ]:
merged["overview"] = merged["overview"].apply(
    lambda x: x.split()
)

In [ ]:
merged["overview"].iloc[0][:20]

['Led',
 'by',
 'Woody,',
 "Andy's",
 'toys',
 'live',
 'happily',
 'in',
 'his',
 'room',
 'until',
 "Andy's",
 'birthday',
 'brings',
 'Buzz',
 'Lightyear',
 'onto',
 'the',
 'scene.',
 'Afraid']

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

merged["rating_score"] = scaler.fit_transform(
    merged[["vote_average"]]
)

In [ ]:
merged["rating_score"].describe()

,rating_score
count,2524.000000
mean,0.477517
std,0.119563
min,0.000000
25%,0.408451
50%,0.478873
75%,0.563380
max,1.000000


In [ ]:

merged["popularity_log"] = np.log1p(merged["popularity"])


merged["popularity_score"] = scaler.fit_transform(
    merged[["popularity_log"]]
)

merged["popularity_score"].describe()

,popularity_score
count,2524.000000
mean,0.421602
std,0.141892
min,0.000000
25%,0.329070
50%,0.432830
75%,0.521502
max,1.000000


In [ ]:

merged["popularity_final"] = (
    0.60 * merged["rating_score"] +
    0.40 * merged["popularity_score"]
)

merged[
    [
        "vote_average",
        "rating_score",
        "popularity",
        "popularity_score",
        "popularity_final"
    ]
].head()

,vote_average,rating_score,popularity,popularity_score,popularity_final
0,7.7,0.676056,73.640445,0.634645,0.659492
1,6.6,0.521127,59.824565,0.604287,0.554391
2,7.1,0.591549,3.770161,0.226730,0.445622
3,5.7,0.394366,7.029308,0.303962,0.358204
4,7.8,0.690141,40.066880,0.546029,0.632496


In [ ]:

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

overview_text = merged["overview"].apply(
    lambda words: " ".join(words)
)

merged["sentiment_score"] = overview_text.apply(
    lambda text: sia.polarity_scores(text)["compound"]
)

# Normalize from [-1,1] to [0,1]
merged["sentiment_score"] = (
    merged["sentiment_score"] + 1
) / 2

In [ ]:
merged["sentiment_score"].describe()

,sentiment_score
count,2524.000000
mean,0.463595
std,0.321110
min,0.002150
25%,0.149262
50%,0.448650
75%,0.778700
max,0.997400


In [ ]:

print(merged.info())

print()

print(merged.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2524 entries, 0 to 2523
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   movieId           2524 non-null   int64  
 1   tmdb_id           2524 non-null   int64  
 2   title_ml          2524 non-null   object 
 3   title_tmdb        2524 non-null   object 
 4   clean_title       2524 non-null   object 
 5   year              2524 non-null   float64
 6   genres_ml         2524 non-null   object 
 7   genres_tmdb       2524 non-null   object 
 8   keywords          2524 non-null   object 
 9   overview          2524 non-null   object 
 10  cast              2524 non-null   object 
 11  crew              2524 non-null   object 
 12  popularity        2524 non-null   float64
 13  vote_average      2524 non-null   float64
 14  vote_count        2524 non-null   int64  
 15  director          2524 non-null   object 
 16  rating_score      2524 non-null   float64


In [ ]:
##MATRICES

In [ ]:

from sklearn.preprocessing import MultiLabelBinarizer

genre_mlb = MultiLabelBinarizer()

genre_matrix = genre_mlb.fit_transform(
    merged["genres_tmdb"]
)

print("Genre Matrix Shape:", genre_matrix.shape)

print()

print("Genres")

print(genre_mlb.classes_)

Genre Matrix Shape: (2524, 20)

Genres
['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama'
 'Family' 'Fantasy' 'Foreign' 'History' 'Horror' 'Music' 'Mystery'
 'Romance' 'Science Fiction' 'TV Movie' 'Thriller' 'War' 'Western']


In [ ]:
genre_df = pd.DataFrame(
    genre_matrix,
    columns=genre_mlb.classes_
)

genre_df.head()

,Action,Adventure,Animation,Comedy,Crime,Documentary,Drama,Family,Fantasy,Foreign,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

keyword_text = merged["keywords"].apply(
    lambda x: " ".join(x)
)

tfidf_keywords = TfidfVectorizer(
    min_df=2
)

keyword_matrix = tfidf_keywords.fit_transform(
    keyword_text
)

print("Keyword Matrix Shape:",
      keyword_matrix.shape)

print()

print("Vocabulary Size:",
      len(tfidf_keywords.get_feature_names_out()))

Keyword Matrix Shape: (2524, 3139)

Vocabulary Size: 3139


In [ ]:
print(genre_matrix.shape)

print(keyword_matrix.shape)

(2524, 20)
(2524, 3139)


In [ ]:

import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def clean_overview(words):

    cleaned = []

    for word in words:

        word = word.lower()

        if word.isalpha():

            if word not in stop_words:

                cleaned.append(
                    lemmatizer.lemmatize(word)
                )

    return " ".join(cleaned)


overview_text = merged["overview"].apply(clean_overview)

print(overview_text.iloc[0])

led toy live happily room birthday brings buzz lightyear onto afraid losing place woody plot circumstance separate buzz woody duo eventually learns put aside


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_overview = TfidfVectorizer(
    max_features=5000,
    min_df=2
)

overview_matrix = tfidf_overview.fit_transform(
    overview_text
)

print("Overview Matrix Shape:",
      overview_matrix.shape)

print()

print("Vocabulary Size:",
      len(tfidf_overview.get_feature_names_out()))

Overview Matrix Shape: (2524, 5000)

Vocabulary Size: 5000


In [ ]:
print(overview_matrix.shape)
print(overview_text.iloc[0])

(2524, 5000)
led toy live happily room birthday brings buzz lightyear onto afraid losing place woody plot circumstance separate buzz woody duo eventually learns put aside


In [ ]:

from sklearn.preprocessing import MultiLabelBinarizer

cast_mlb = MultiLabelBinarizer()

cast_matrix = cast_mlb.fit_transform(
    merged["cast"]
)

print("Cast Matrix Shape:", cast_matrix.shape)

print()

print("Unique Actors:",
      len(cast_mlb.classes_))

Cast Matrix Shape: (2524, 3042)

Unique Actors: 3042


In [ ]:
cast_df = pd.DataFrame(
    cast_matrix,
    columns=cast_mlb.classes_
)

cast_df.head()

,"""WeirdAl""Yankovic",50Cent,A.J.Cook,Aaliyah,AamirKhan,AaranThomas,AaronAbrams,AaronEckhart,AaronPaul,AaronRuell,...,ZakOrth,ZanaMarjanović,ZenaGrey,ZhangZiyi,ZoeSaldana,ZoeSloane,ZooeyDeschanel,ZoëBell,ZubaidaSahar,ZuleikhaRobinson
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:

director_mlb = MultiLabelBinarizer()

director_matrix = director_mlb.fit_transform(
    merged["director"].apply(lambda x: [x])
)

print("Director Matrix Shape:",
      director_matrix.shape)

print()

print("Unique Directors:",
      len(director_mlb.classes_))

Director Matrix Shape: (2524, 1250)

Unique Directors: 1250


In [ ]:
director_df = pd.DataFrame(
    director_matrix,
    columns=director_mlb.classes_
)

director_df.head()

,AaronHann,AdamBrooks,AdamMcKay,AdamRifkin,AdamShankman,AdrianLyne,AdrienneShelly,AgnieszkaHolland,AkivaGoldsman,AkivaSchaffer,...,WolfgangBecker,WolfgangPetersen,WongKar-wai,WoodyAllen,WychKaosayananda,XavierGens,ZachBraff,ZackSnyder,ZalBatmanglij,ÀlexPastor
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
print(cast_matrix.shape)

print(director_matrix.shape)

(2524, 3042)
(2524, 1250)


In [ ]:

from sklearn.preprocessing import normalize

genre_matrix_norm = normalize(
    genre_matrix,
    norm="l2"
)

keyword_matrix_norm = normalize(
    keyword_matrix,
    norm="l2"
)

overview_matrix_norm = normalize(
    overview_matrix,
    norm="l2"
)

cast_matrix_norm = normalize(
    cast_matrix,
    norm="l2"
)

director_matrix_norm = normalize(
    director_matrix,
    norm="l2"
)

print("All feature matrices normalized successfully.")

All feature matrices normalized successfully.


In [ ]:

genre_weight = 0.15
keyword_weight = 0.25
overview_weight = 0.35
cast_weight = 0.15
director_weight = 0.10

genre_matrix_weighted = genre_matrix_norm * genre_weight

keyword_matrix_weighted = keyword_matrix_norm * keyword_weight

overview_matrix_weighted = overview_matrix_norm * overview_weight

cast_matrix_weighted = cast_matrix_norm * cast_weight

director_matrix_weighted = director_matrix_norm * director_weight

print("Feature weights applied successfully.")

Feature weights applied successfully.


In [ ]:

from scipy.sparse import hstack

content_matrix = hstack(
    [
        genre_matrix_weighted,
        keyword_matrix_weighted,
        overview_matrix_weighted,
        cast_matrix_weighted,
        director_matrix_weighted
    ]
)

print("Combined Matrix Shape:", content_matrix.shape)

Combined Matrix Shape: (2524, 12451)


In [ ]:
print(content_matrix.shape)

(2524, 12451)


In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(content_matrix)

print("Similarity Matrix Shape:", similarity.shape)

Similarity Matrix Shape: (2524, 2524)


In [ ]:

def content_recommend(movie_name, top_n=10):

    movie_name = movie_name.lower().strip()

    idx = merged[
        merged["title_tmdb"].str.lower() == movie_name
    ].index

    if len(idx) == 0:
        print("Movie not found!")
        return []

    idx = idx[0]

    similarity_scores = list(
        enumerate(similarity[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:top_n+1]

    recommendations = []

    for movie_idx, score in similarity_scores:

        recommendations.append(
            (
                merged.iloc[movie_idx]["title_tmdb"],
                score
            )
        )

    return recommendations

In [ ]:
recommendations = content_recommend("Toy Story")

for movie, score in recommendations:
    print(f"{movie} --> {score:.3f}")

Toy Story 2 --> 0.428
Toy Story 3 --> 0.341
Small Soldiers --> 0.244
Pinocchio --> 0.177
Monster House --> 0.155
Monsters, Inc. --> 0.153
Big --> 0.146
Drive Me Crazy --> 0.146
Ted 2 --> 0.142
Ted --> 0.138


In [ ]:
recommendations = content_recommend("Casino")

for movie, score in recommendations:
    print(f"{movie} --> {score:.3f}")

Wild Card --> 0.282
Taxi Driver --> 0.227
Lucky You --> 0.207
Things to Do in Denver When You're Dead --> 0.188
Requiem for a Dream --> 0.181
Showgirls --> 0.180
Bringing Out the Dead --> 0.176
Donnie Brasco --> 0.175
Raging Bull --> 0.170
Last Vegas --> 0.168


In [ ]:
recommendations = content_recommend("GoldenEye")

for movie, score in recommendations:
    print(f"{movie} --> {score:.3f}")

You Only Live Twice --> 0.283
Live and Let Die --> 0.281
Dr. No --> 0.278
Octopussy --> 0.277
Die Another Day --> 0.265
From Russia with Love --> 0.263
Licence to Kill --> 0.250
On Her Majesty's Secret Service --> 0.242
Tomorrow Never Dies --> 0.237
Never Say Never Again --> 0.232


In [ ]:
print(similarity.shape)

(2524, 2524)


In [ ]:

from surprise import Dataset
from surprise import Reader

reader = Reader(
    rating_scale=(0.5, 5)
)

data = Dataset.load_from_df(
    ratings[
        ["userId", "movieId", "rating"]
    ],
    reader
)

print("Ratings dataset prepared successfully.")

Ratings dataset prepared successfully.


In [ ]:

from surprise.model_selection import train_test_split

trainset, testset = train_test_split(
    data,
    test_size=0.20,
    random_state=42
)

print("Training Ratings :", trainset.n_ratings)

print("Testing Ratings :", len(testset))

Training Ratings : 84271
Testing Ratings : 21068


In [ ]:

from surprise import SVD

svd = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd.fit(trainset)

print("SVD model trained successfully.")

SVD model trained successfully.


In [ ]:
print(trainset.n_ratings)

print(len(testset))

84271
21068


In [ ]:
from surprise import accuracy

predictions = svd.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print("RMSE:", round(rmse, 4))
print("MAE :", round(mae, 4))

RMSE: 0.8671
MAE:  0.6719
RMSE: 0.8671
MAE : 0.6719


In [ ]:

def predict_rating(user_id, movie_id):
    prediction = svd.predict(user_id, movie_id)
    return prediction.est

In [ ]:
test_movies = [1, 10, 16, 32, 47]

for movie in test_movies:
    predicted = predict_rating(1, movie)
    movie_name = movies[movies["movieId"] == movie]["title"].values[0]
    print(f"{movie_name:35} {predicted:.2f}")

Toy Story (1995)                    3.65
GoldenEye (1995)                    3.41
Casino (1995)                       3.78
Twelve Monkeys (a.k.a. 12 Monkeys) (1995) 4.14
Seven (a.k.a. Se7en) (1995)         4.17


In [ ]:

def hybrid_recommend(movie_name, user_id=1, top_n=10):

    movie_name = movie_name.lower().strip()

    # Partial match instead of exact match
    matches = merged[
        merged["title_tmdb"].str.lower().str.contains(movie_name, na=False)
    ]

    if len(matches) == 0:
        print("Movie not found!")
        return []

    idx = matches.index[0]

    similarity_scores = list(enumerate(similarity[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:101]

    recommendations = []

    for movie_idx, content_score in similarity_scores:

        movie = merged.iloc[movie_idx]

        movie_id = movie["movieId"]

        collaborative_score = predict_rating(
            user_id,
            movie_id
        ) / 5.0

        popularity_score = movie["popularity_final"]

        rating_score = movie["rating_score"]

        sentiment_score = movie["sentiment_score"]

        final_score = (
            0.50 * content_score +
            0.25 * collaborative_score +
            0.15 * popularity_score +
            0.05 * rating_score +
            0.05 * sentiment_score
        )

        recommendations.append(
            (
                movie["title_tmdb"],
                final_score,
                content_score,
                collaborative_score,
                popularity_score
            )
        )

    recommendations.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations[:top_n]

In [ ]:

def find_movie(query, top_n=10):

    query = query.lower().strip()

    matches = merged[
        merged["title_tmdb"].str.lower().str.contains(query, na=False)
    ][["title_tmdb", "year"]]

    matches = matches.drop_duplicates()

    if len(matches) == 0:
        print("No movies found!")
        return None

    print(f"\nFound {len(matches)} matching movies:\n")

    for i, (_, row) in enumerate(matches.head(top_n).iterrows(), start=1):
        print(f"{i}. {row['title_tmdb']} ({int(row['year'])})")

    return matches

In [ ]:
find_movie("toy")


Found 3 matching movies:

1. Toy Story (1995)
2. Toy Story 2 (1999)
3. Toy Story 3 (2010)


,title_tmdb,year
0,Toy Story,1995.0
614,Toy Story 2,1999.0
1997,Toy Story 3,2010.0


In [ ]:
find_movie("dark")


Found 18 matching movies:

1. Under Siege 2: Dark Territory (1995)
2. Dark City (1998)
3. Dancer in the Dark (2000)
4. Donnie Darko (2001)
5. Darkness Falls (2003)
6. Darkness (2002)
7. Alone in the Dark (2005)
8. Dark Water (2005)
9. Taxi to the Dark Side (2007)
10. Edge of Darkness (2010)


,title_tmdb,year
52,Under Siege 2: Dark Territory,1995.0
322,Dark City,1998.0
745,Dancer in the Dark,2000.0
921,Donnie Darko,2001.0
1099,Darkness Falls,2003.0
1366,Darkness,2002.0
1387,Alone in the Dark,2005.0
1429,Dark Water,2005.0
1725,Taxi to the Dark Side,2007.0
1946,Edge of Darkness,2010.0


In [ ]:
hybrid_recommend("toy")

[('Toy Story 2',
  np.float64(0.5792245752965068),
  np.float64(0.4276182619656949),
  np.float64(0.7845058121911972),
  np.float64(0.6256371718193485)),
 ('Toy Story 3',
  np.float64(0.5150472580410245),
  np.float64(0.34129688344045755),
  np.float64(0.7498624006008672),
  np.float64(0.639064164141887)),
 ('Pinocchio',
  np.float64(0.4406165195756788),
  np.float64(0.17690038138180275),
  np.float64(0.7676657863036731),
  np.float64(0.5644557881623471)),
 ('Monsters, Inc.',
  np.float64(0.41210415366187936),
  np.float64(0.15258139901469442),
  np.float64(0.7877220128809999),
  np.float64(0.6644072315806605)),
 ('Big Hero 6',
  np.float64(0.40791386585696365),
  np.float64(0.0924143411487365),
  np.float64(0.6876956000804376),
  np.float64(0.7278050200597658)),
 ('Ratatouille',
  np.float64(0.39217297163033654),
  np.float64(0.08885793976376097),
  np.float64(0.6827100329370082),
  np.float64(0.635897515446806)),
 ('Inside Out',
  np.float64(0.3911291663507909),
  np.float64(0.088954

In [ ]:


import pickle

# Similarity Matrix
pickle.dump(
    similarity,
    open("similarity.pkl", "wb")
)

# Final Dataset
pickle.dump(
    merged,
    open("movies.pkl", "wb")
)

# Trained SVD Model
pickle.dump(
    svd,
    open("svd_model.pkl", "wb")
)

# Genre Encoder
pickle.dump(
    genre_mlb,
    open("genre_mlb.pkl", "wb")
)

# Cast Encoder
pickle.dump(
    cast_mlb,
    open("cast_mlb.pkl", "wb")
)

# Director Encoder
pickle.dump(
    director_mlb,
    open("director_mlb.pkl", "wb")
)

# Keyword TF-IDF
pickle.dump(
    tfidf_keywords,
    open("tfidf_keywords.pkl", "wb")
)

# Overview TF-IDF
pickle.dump(
    tfidf_overview,
    open("tfidf_overview.pkl", "wb")
)

print("All models saved successfully!")

All models saved successfully!


In [ ]:
import os
import shutil

model_files = [
    "movies.pkl",
    "similarity.pkl",
    "svd_model.pkl",
    "genre_mlb.pkl",
    "cast_mlb.pkl",
    "director_mlb.pkl",
    "tfidf_keywords.pkl",
    "tfidf_overview.pkl"
]

save_path = "/content/drive/MyDrive/movie_recommender_models"

os.makedirs(save_path, exist_ok=True)

for file in model_files:
    shutil.copy(file, save_path)

print("✅ All models saved to Google Drive!")

✅ All models saved to Google Drive!


In [ ]:
import os

os.listdir("/content/drive/MyDrive/movie_recommender_models")

['movies.pkl',
 'similarity.pkl',
 'svd_model.pkl',
 'genre_mlb.pkl',
 'cast_mlb.pkl',
 'director_mlb.pkl',
 'tfidf_keywords.pkl',
 'tfidf_overview.pkl']

In [ ]:

import pickle

movie_recommender = {


    "movies": merged,

    "similarity": similarity,


    "svd": svd,


    "genre_mlb": genre_mlb,
    "cast_mlb": cast_mlb,
    "director_mlb": director_mlb,


    "tfidf_keywords": tfidf_keywords,
    "tfidf_overview": tfidf_overview

}

with open("movie_recommender_model.pkl", "wb") as f:
    pickle.dump(movie_recommender, f)

print("✅ Complete recommender model saved successfully!")

✅ Complete recommender model saved successfully!


In [ ]:
import os

print(os.path.exists("movie_recommender_model.pkl"))

True


In [ ]:

with open("movie_recommender_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print(type(loaded_model))

print()

print(loaded_model.keys())

<class 'dict'>

dict_keys(['movies', 'similarity', 'svd', 'genre_mlb', 'cast_mlb', 'director_mlb', 'tfidf_keywords', 'tfidf_overview'])


In [ ]:
import os
import shutil

save_path = "/content/drive/MyDrive/movie_recommender"

os.makedirs(save_path, exist_ok=True)

shutil.copy(
    "movie_recommender_model.pkl",
    save_path
)

print("✅ Model copied to Google Drive!")

✅ Model copied to Google Drive!


In [ ]:
os.listdir("/content/drive/MyDrive/movie_recommender")

['movie_recommender_model.pkl']

In [ ]:


movies_loaded = loaded_model["movies"]
similarity_loaded = loaded_model["similarity"]
svd_loaded = loaded_model["svd"]

print(movies_loaded.shape)

print(similarity_loaded.shape)

print(svd_loaded.predict(1, 1).est)

(2524, 21)
(2524, 2524)
3.654806614894816


In [ ]:
import importlib.metadata

print(importlib.metadata.version("python-dotenv"))

1.2.2
